# Courtship song — paper-ready figures

Per paired courtship bout this notebook answers four questions:

1. Which fly is the male? (the singer — Drosophila courtship song is male-produced)
2. What song types is he performing? (pulse / sine / waggle / quiet)
3. What are his walking patterns during singing? (forward speed, turn rate, walking vs stopped)
4. How does his center-of-mass height change across song types?

Data source: the post-rescue relinked h5 produced by `rescue_identity_relink_inplace.py`.
All reusable logic lives in `utils/song_analysis.py`, `utils/sex_id.py`, `utils/locomotion.py`.
The sandbox notebook `Courtship_Song_Analysis.ipynb` is untouched; this is a clean sibling.


In [ ]:
# --- Setup: imports + matplotlib style + output dir ---
from __future__ import annotations

import os, sys, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Project imports (notebook runs from the repo root OR from notebooks/)
REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'utils').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.courtship_loader import (
    load_courtship_h5, pair_bouts, get_fields, analyze_pair, analyze_all_pairs,
)
from utils.pair_validity import PairValidityConfig
from utils.song_analysis import SongAnalysisConfig
from utils.sex_id import SexIdConfig
from utils.locomotion import LocomotionConfig
from utils.locomotion import walking_fraction

# Paper-ready matplotlib defaults: Type-1 / sans-serif / clean spines
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
})
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Colors shared across figures
SONG_COLORS = {
    'pulse':  '#d62728',  # red
    'sine':   '#1f77b4',  # blue
    'waggle': '#9467bd',  # purple
    'quiet':  '#bdbdbd',  # grey
}
# After the reorder in analyze_pair, slot 0 = male, slot 1 = female.
FLY_COLORS = {'fly0': '#ff7f0e', 'fly1': '#2ca02c'}  # male, female

FIG_DIR = REPO_ROOT / 'figures' / 'courtship'
FIG_DIR.mkdir(parents=True, exist_ok=True)

H5_PATH = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/04092026_bouts_04112026/'
               'v1/'
               'ik_output_combined_v1_courtship_both.h5')

CACHE_PATH = FIG_DIR / 'per_bout_cache.pkl'

print(f'repo   : {REPO_ROOT}')
print(f'h5     : {H5_PATH}')
print(f'figures: {FIG_DIR}')


In [ ]:
from utils.courtship_loader import load_and_merge_courtship_h5
# --- Load the relinked h5 and pair fly0/fly1 bouts ---
# Each bout key in this h5 holds ONE fly's kp_data / xpos_egocentric / qpos.
# The pairing is consecutive (fly0, fly1) and validated by info['source_flies'].

print('loading h5 (slow — full file)...')
# data, info, kp_names, bout_keys = load_courtship_h5(H5_PATH)

H5_SESSION0 = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/'
                   'Session0/2025_10_20_13_20_04/Predictions_3D_34643097/'
                   'Predictions_3D_34653429/v1/ik_output_combined_v1_courtship_both.h5')
H5_MAIN = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/'
               '04092026_bouts_04112026/v1/ik_output_combined_v1_courtship_both.h5')

data, info, kp_names, bout_keys = load_and_merge_courtship_h5([H5_SESSION0, H5_MAIN])


pairs = pair_bouts(bout_keys, info)
print(f'bouts: {len(bout_keys)}  pairs: {len(pairs)}  kp: {len(kp_names)}')
print(f'first 3 pairs: {pairs[:3]}')


In [ ]:
# --- Run the full per-pair analysis and cache to a pickle ---
# Set FORCE=True to invalidate the cache and re-run from scratch.
FORCE = True

song_cfg = SongAnalysisConfig()
sex_cfg  = SexIdConfig()
loc_cfg  = LocomotionConfig()
pair_cfg = PairValidityConfig()

results = analyze_all_pairs(
    data, pairs, kp_names,
    cache_path=CACHE_PATH,
    force=FORCE,
    song_cfg=song_cfg,
    sex_cfg=sex_cfg,
    loc_cfg=loc_cfg,
    pair_cfg=pair_cfg,
)

print(f'ok: {len(results)} pair results loaded')


In [ ]:
# --- Build a per-bout results dataframe for figures + CSV export ---
def _frac(labels, valid):
    labels = np.asarray(labels); valid = np.asarray(valid, dtype=bool)
    n = int(valid.sum())
    if n == 0:
        return {k: float('nan') for k in ('pulse', 'sine', 'waggle', 'quiet')}
    sel = labels[valid]
    return {k: float(np.mean(sel == k)) for k in ('pulse', 'sine', 'waggle', 'quiet')}

rows = []
for r in results:
    sex = r['sex']
    fr  = _frac(r['male_labels'], r['male_valid'])
    bs  = r['by_song']
    row = {
        'pair_idx':                   r['pair_idx'],
        # Post-swap keys: key0 is always the male's original bout, key1 the female's.
        'key0':                       r['key0'],
        'key1':                       r['key1'],
        'T':                          r['T'],
        'male_id':                    sex['male_id'],
        'female_id':                  sex['female_id'],
        'criterion':                  sex['criterion'],
        'confidence':                 sex['confidence'],
        'disagree_bodylen':           sex['disagree'],
        # Semantic (post-swap) song fractions. slot0=male, slot1=female.
        'song_fraction_male':         r['song0']['summary']['song_fraction'],
        'song_fraction_female':       r['song1']['summary']['song_fraction'],
        'body_length_male':           sex['body_length_male'],
        'body_length_female':         sex['body_length_female'],
        # Raw tracker-slot record for traceability + Figure 1.
        'tracker_male_id':            r['tracker_male_id'],
        'tracker_key0':               r['tracker_key0'],
        'tracker_key1':               r['tracker_key1'],
        'tracker_song_fraction_fly0': r['tracker_song_fraction_fly0'],
        'tracker_song_fraction_fly1': r['tracker_song_fraction_fly1'],
        'frac_pulse':                 fr['pulse'],
        'frac_sine':                  fr['sine'],
        'frac_waggle':                fr['waggle'],
        'frac_quiet':                 fr['quiet'],
        'walking_fraction':           walking_fraction(r['walking_state'][r['male_valid']]) if r['male_valid'].any() else float('nan'),
        'mean_com_z':                 float(np.nanmean(r['com_z'][r['male_valid']])) if r['male_valid'].any() else float('nan'),
        'mean_speed_bl':              float(np.nanmean(r['kin'].get('speed_bl', r['kin']['speed'])[r['male_valid']])) if r['male_valid'].any() else float('nan'),
    }
    # Song-conditioned aggregates
    for song in ('pulse', 'sine', 'waggle', 'quiet'):
        stats = bs.get(song, {})
        for m in ('speed_bl', 'forward_speed_bl', 'turn_rate', 'com_z'):
            row[f'{song}_{m}_mean'] = stats.get(m, {}).get('mean', float('nan'))
            row[f'{song}_{m}_n']    = stats.get(m, {}).get('n',    0)
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head()


In [ ]:
%matplotlib inline

In [ ]:
# # --- Figure 1: Male/female identification ---
# fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.3))

# # Panel A plots the raw tracker-slot song fractions and colours by which
# # tracker slot was assigned male (pre-swap). The rest of the notebook uses
# # the post-swap convention where slot-0 is always the male.
# sf0 = df['tracker_song_fraction_fly0'].values
# sf1 = df['tracker_song_fraction_fly1'].values
# male_is_fly0 = (df['tracker_male_id'] == 'fly0').values

# # Panel A — song-fraction scatter
# ax = axes[0]
# ax.scatter(sf0[male_is_fly0],  sf1[male_is_fly0],  s=10,
#            c=FLY_COLORS['fly0'], alpha=0.75, label='male = fly0',
#            edgecolor='none')
# ax.scatter(sf0[~male_is_fly0], sf1[~male_is_fly0], s=10,
#            c=FLY_COLORS['fly1'], alpha=0.75, label='male = fly1',
#            edgecolor='none')
# ax.plot([0, 1], [0, 1], '--', color='k', lw=0.6, alpha=0.5)
# ax.set_xlim(0, max(0.05, float(np.nanmax(sf0)) * 1.05))
# ax.set_ylim(0, max(0.05, float(np.nanmax(sf1)) * 1.05))
# ax.set_xlabel('song fraction (fly0)')
# ax.set_ylabel('song fraction (fly1)')
# ax.set_title('A  Song-based male ID')
# ax.legend(frameon=False, loc='upper right')

# # Panel B — body length by assigned sex
# ax = axes[1]
# bl_m = df['body_length_male'].values
# bl_f = df['body_length_female'].values
# bl_m = bl_m[np.isfinite(bl_m)]
# bl_f = bl_f[np.isfinite(bl_f)]
# parts = ax.violinplot([bl_m, bl_f], positions=[0, 1], widths=0.8,
#                       showmeans=True, showextrema=False)
# for pc, col in zip(parts['bodies'], ['#377eb8', '#e41a1c']):
#     pc.set_facecolor(col); pc.set_alpha(0.6); pc.set_edgecolor('k'); pc.set_linewidth(0.5)
# ax.set_xticks([0, 1]); ax.set_xticklabels(['male', 'female'])
# ax.set_ylabel('body length (data units)')
# ax.set_title('B  Body length by assigned sex')

# # Panel C — confidence histogram
# ax = axes[2]
# ax.hist(df['confidence'].values, bins=np.linspace(0, 1, 21),
#         color='#555555', edgecolor='white', linewidth=0.4)
# ax.axvline(0.5, color='k', lw=0.5, ls='--')
# ax.set_xlabel('assignment confidence')
# ax.set_ylabel('# pairs')
# ax.set_title('C  Male ID confidence')

# plt.tight_layout()
# out = FIG_DIR / 'fig1_male_female_id.pdf'
# plt.savefig(out)
# print(f'saved {out}')
# plt.show()


In [ ]:
# --- Figure 2: Song classification example bout (rich 3-panel both-wings view) ---
# Pick a specific pair to visualise. Default is pair 88 (bout_176/bout_177):
#   long sine passage 0.0-0.464s, followed by alternating pulse bursts.
# Set EXAMPLE_IDX=None to auto-pick the bout with the most pulse+sine content.
EXAMPLE_IDX = 13 # 73

# Optional sub-range (in frames, bout-relative) to zoom into a clip of
# the example bout. Inherited by Figure 3 below. None = full bout.
# When trimmed, plotted arrays are sliced so y-axes auto-scale to the
# in-range data only (extreme values outside the range are dropped).
START_FRAME = (150/1000) * 800  # inclusive
END_FRAME   = (1500/1000) * 800  # exclusive; None → render to end of bout

if EXAMPLE_IDX is None:
    example_idx = int((df['frac_pulse'] + df['frac_sine']).idxmax())
else:
    example_idx = int(EXAMPLE_IDX)
ex = results[example_idx]
print(f'example pair_idx={example_idx}  keys={ex["key0"]}/{ex["key1"]}  '
      f'T={ex["T"]} ({ex["T"]/song_cfg.fs:.3f}s)  male={ex["sex"]["male_id"]}')

fs = song_cfg.fs
male_song = ex['song0'] if ex['sex']['male_id'] == 'fly0' else ex['song1']

# ----- plotting helpers (local — not exported) ----------------------------
_SEG_COLORS = {
    'pulse':  '#E76F5133',
    'sine':   '#2A9D8F33',
    'waggle': '#E9C46A33',
    'quiet':  '#cccccc11',
}
_SEG_EDGE_COLORS = {
    'pulse':  '#E76F51',
    'sine':   '#2A9D8F',
    'waggle': '#E9C46A',
    'quiet':  '#999999',
}
_WING_COLORS = {
    'WingL_V12': '#7B2D8E', 'WingL_V13': '#A855F7',
    'WingR_V12': '#0369A1', 'WingR_V13': '#38BDF8',
}


def _add_segment_shading(ax, segments, fs, hatch=None, side_tag=None,
                         seen_labels=None, frame_range=None):
    if seen_labels is None:
        seen_labels = set()
    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        # Clip to the visible frame range if provided; drop segments that
        # fall entirely outside so they don't pollute the legend.
        if frame_range is not None:
            lo, hi = frame_range
            if seg['end'] <= lo or seg['start'] >= hi:
                continue
            seg_start = max(seg['start'], lo)
            seg_end   = min(seg['end'],   hi)
        else:
            seg_start = seg['start']
            seg_end   = seg['end']
        s_ms = seg_start / fs * 1000.0
        e_ms = seg_end   / fs * 1000.0
        color = _SEG_COLORS.get(stype, '#cccccc33')
        label_key = f'{stype}_{side_tag}'
        if label_key in seen_labels:
            label = None
        else:
            label = f'{stype.capitalize()} {side_tag}' if side_tag else stype.capitalize()
            seen_labels.add(label_key)
        ax.axvspan(s_ms, e_ms, facecolor=color, zorder=0, label=label,
                   hatch=hatch, edgecolor=_SEG_EDGE_COLORS.get(stype),
                   linewidth=0)


def _add_segment_freq_annotations(ax, segments, window_features, fs,
                                  frame_range=None):
    wc = window_features.get('window_centers')
    pf = window_features.get('peak_freq')
    if wc is None or pf is None or len(wc) == 0:
        return
    wc = np.asarray(wc); pf = np.asarray(pf)
    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        s, e = seg['start'], seg['end']
        if frame_range is not None:
            lo, hi = frame_range
            if e <= lo or s >= hi:
                continue
            s = max(s, lo); e = min(e, hi)
        in_seg = (wc >= s) & (wc <= e)
        if in_seg.sum() == 0:
            continue
        seg_freqs = pf[in_seg]
        median_f = float(np.nanmedian(seg_freqs))
        min_f = float(np.nanmin(seg_freqs))
        max_f = float(np.nanmax(seg_freqs))
        mid_ms = (s + e) / 2.0 / fs * 1000.0
        edge_color = _SEG_EDGE_COLORS.get(stype, '#999999')
        freq_str = f'{median_f:.0f}Hz' if min_f == max_f else f'{min_f:.0f}-{max_f:.0f}Hz'
        ax.annotate(
            freq_str,
            xy=(mid_ms, 0.02), xycoords=('data', 'axes fraction'),
            fontsize=7, color=edge_color, fontweight='bold',
            ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=edge_color,
                      alpha=0.85, lw=0.7),
        )


def _shade_song(ax, result, fs, seen, frame_range=None):
    # L and R carry identical labels (bilateral pulse mask overwrites both),
    # so a single shading pass is enough — using L's segments.
    _add_segment_shading(ax, result['sides']['L']['segments'], fs=fs,
                         hatch=None, side_tag=None, seen_labels=seen,
                         frame_range=frame_range)


# ----- figure --------------------------------------------------------------
T = male_song['summary']['n_frames']
t_ms_full = np.arange(T) / fs * 1000.0

# Resolve optional sub-range (bout-relative frame indices).
_sf = 0 if START_FRAME is None else max(0, int(START_FRAME))
_ef = T if END_FRAME   is None else min(T, int(END_FRAME))
if _ef <= _sf:
    raise ValueError(f'END_FRAME ({END_FRAME}) must be greater than START_FRAME ({START_FRAME})')
_trimmed = not (_sf == 0 and _ef == T)
_frame_range = (_sf, _ef) if _trimmed else None
clip_tag = f'_f{_sf}-{_ef}' if _trimmed else ''

# Sliced time axis (absolute, in ms) — plots use this so y-autoscale
# only sees the in-range samples.
t_ms = t_ms_full[_sf:_ef]

dom = male_song['dominant_wing']
dom_side = male_song['sides'][dom]
wf = dom_side['window_features']
s = dom_side['summary']

mode_parts = []
for k, lab in (('frac_pulse', 'pulse'), ('frac_sine', 'sine'),
               ('frac_waggle', 'waggle'), ('frac_quiet', 'quiet')):
    v = s.get(k, 0.0)
    if v > 0:
        mode_parts.append(f'{lab}={100*v:.0f}%')
mode_str = '  '.join(mode_parts)

# Build rows dynamically: always show Z and Y panels; show qpos panel only
# if joint angles are available for this bout.
joints = male_song.get('joints')
nrows = 3 if joints is not None else 2
fig, axes = plt.subplots(
    nrows, 1, figsize=(10.0, 3.0 * nrows), sharex=True,
    gridspec_kw={'height_ratios': [1.2, 1.0] + ([1.2] if nrows == 3 else []),
                 'hspace': 0.28},
)
if nrows == 1:
    axes = [axes]
_range_str = f'  |  frames [{_sf}, {_ef})' if _trimmed else ''
fig.suptitle(
    f'pair {example_idx}  ({ex["key0"]}/{ex["key1"]})  |  '
    f'{male_song["summary"]["duration_s"]:.3f}s  |  male={ex["sex"]["male_id"]}  |  '
    f'dom_wing={dom}  |  {mode_str}{_range_str}',
    fontsize=10, fontweight='bold', y=0.995,
)

wd = male_song['wing_data']

# ===== PANEL 1: Wing Tip Z (V13, both wings overlaid) =====
ax = axes[0]
seen = set()
_shade_song(ax, male_song, fs, seen, frame_range=_frame_range)
ax.plot(t_ms, np.asarray(wd['WingL_V13']['z'])[_sf:_ef],
        color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
ax.plot(t_ms, np.asarray(wd['WingR_V13']['z'])[_sf:_ef],
        color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
_add_segment_freq_annotations(ax, dom_side['segments'], wf, fs,
                              frame_range=_frame_range)
ax.set_ylabel('Z (world)')
ax.set_title('Wing tip Z (V13)')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
ax.grid(True, alpha=0.3)

# ===== PANEL 2: Wing Tip Y (V13, both wings overlaid) =====
ax = axes[1]
seen = set()
_shade_song(ax, male_song, fs, seen, frame_range=_frame_range)
ax.plot(t_ms, np.asarray(wd['WingL_V13']['y'])[_sf:_ef],
        color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
ax.plot(t_ms, np.asarray(wd['WingR_V13']['y'])[_sf:_ef],
        color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
ax.set_ylabel('Y (world)')
ax.set_title('Wing tip Y (V13)')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
ax.grid(True, alpha=0.3)

# ===== PANEL 3 (optional): qpos wing joint angles =====
if joints is not None:
    ax = axes[2]
    seen = set()
    _shade_song(ax, male_song, fs, seen, frame_range=_frame_range)
    joint_styles = [
        ('yaw_L',   '#7B2D8E', '-',  'L yaw'),
        ('roll_L',  '#7B2D8E', '--', 'L roll'),
        ('pitch_L', '#7B2D8E', ':',  'L pitch'),
        ('yaw_R',   '#0369A1', '-',  'R yaw'),
        ('roll_R',  '#0369A1', '--', 'R roll'),
        ('pitch_R', '#0369A1', ':',  'R pitch'),
    ]
    for key, color, ls, label in joint_styles:
        if key in joints:
            ax.plot(t_ms,
                    np.degrees(np.asarray(joints[key]))[_sf:_ef],
                    color=color, ls=ls, lw=1.0, alpha=0.85, label=label)
    ax.set_ylabel('angle (deg)')
    ax.set_title('Wing joint angles (qpos)')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('time (ms)')

# Overlay peak-detector pulse events as vertical dashed lines, filtered
# to the visible frame range so they don't clutter the legend count.
pef_full = np.asarray(wf.get('pulse_event_frames', np.array([], dtype=int)))
if pef_full.size > 0:
    pef = pef_full[(pef_full >= _sf) & (pef_full < _ef)]
    pef_ms = pef / fs * 1000.0
    for _ax in axes:
        for i, xm in enumerate(pef_ms):
            _ax.axvline(xm, color='#E76F51', lw=0.6, alpha=0.55,
                        ls='--', zorder=4,
                        label=(f'pulse events (n={len(pef_ms)})'
                               if i == 0 and _ax is axes[0] else None))
    axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7)

# Tighten x-limits to the trimmed range (redundant after slicing, but
# pins the view if axvspans would otherwise stretch it).
for _ax in axes:
    _ax.set_xlim(t_ms_full[_sf], t_ms_full[min(_ef - 1, T - 1)])

# (tight_layout skipped: external legends use bbox_to_anchor)
out = FIG_DIR / f'fig2_song_example_pair{example_idx:03d}{clip_tag}.pdf'
# plt.savefig(out)
# print(f'saved {out}')
plt.show()


In [ ]:
# --- Diagnostic: why is ~225-375 ms of pair 23 labeled 'pulse' instead of 'sine'? ---
# Reconstructs the peak-override stage of _bilateral_pulse_mask so we can see
# whether the misclassification is coming from (a) spurious peaks surviving
# the baseline-ratio gate inside the sine region, (b) real pulses outside the
# region being bridged through it by pulse_train_max_gap_ms, or (c) the FFT
# stage calling it pulse before the peak override runs.
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

_cfg   = song_cfg
_fs    = _cfg.fs
_dom   = male_song['dominant_wing']
_side  = male_song['sides'][_dom]
_wd    = male_song['wing_data']
z_L    = np.asarray(_wd[_cfg.left_tip ]['z'], dtype=float)
z_R    = np.asarray(_wd[_cfg.right_tip]['z'], dtype=float)
T      = len(z_L)
t_ms_  = np.arange(T) / _fs * 1000.0

# bilateral |dZ/dt| exactly as the classifier builds it
_dzL    = np.abs(np.diff(z_L, prepend=z_L[0]) * _fs)
_dzR    = np.abs(np.diff(z_R, prepend=z_R[0]) * _fs)
dz_max  = np.maximum(_dzL, _dzR)

_min_dist = _cfg.ms_to_frames(_cfg.pulse_detect_min_dist_ms)
_cands, _ = find_peaks(
    dz_max,
    height=_cfg.pulse_detect_height,
    prominence=_cfg.pulse_detect_prominence,
    distance=_min_dist,
)
_kept = set(np.asarray(_side['window_features'].get('pulse_event_frames', []), dtype=int))
_rejected = np.array([p for p in _cands if int(p) not in _kept], dtype=int)
_kept_arr = np.array(sorted(_kept), dtype=int)

# Recompute baseline ratio for every candidate (same recipe as lines 489-506)
_w_base = _cfg.ms_to_frames(_cfg.pulse_baseline_window_ms)
_ex     = _cfg.ms_to_frames(_cfg.pulse_baseline_exclude_ms)
def _baseline_ratio(p):
    lo, hi = max(0, p - _w_base), min(T, p + _w_base + 1)
    local = dz_max[lo:hi]
    clo = max(0, (p - _ex) - lo); chi = min(len(local), (p + _ex + 1) - lo)
    local = np.concatenate([local[:clo], local[chi:]])
    if len(local) == 0:
        return np.nan
    return dz_max[p] / max(float(np.median(local)), 1e-9)
_ratio = {int(p): _baseline_ratio(int(p)) for p in _cands}

_LO_MS, _HI_MS = 225.0, 375.0
_lo_f = int(round(_LO_MS * _fs / 1000.0))
_hi_f = int(round(_HI_MS * _fs / 1000.0))
print(f'Pair {example_idx}  keys={ex["key0"]}/{ex["key1"]}  T={T}  fs={_fs:.0f}')
print(f'Problem window {_LO_MS:.0f}-{_HI_MS:.0f} ms  = frames [{_lo_f}, {_hi_f})')
_uniq, _cnt = np.unique(_side['frame_labels'][_lo_f:_hi_f], return_counts=True)
print(f'Final labels over problem window (side {_dom}): {dict(zip(_uniq, _cnt))}')
print()
print('Kept peaks (survived baseline-ratio gate):')
for p in sorted(_kept):
    m = p / _fs * 1000.0
    flag = '  <-- INSIDE sine region' if _lo_f <= p < _hi_f else ''
    print(f'  frame {p:4d}  t={m:7.1f} ms  dz={dz_max[p]:7.2f}  '
          f'ratio={_ratio.get(int(p), float("nan")):5.2f}{flag}')
print()
print('Rejected peaks (failed baseline-ratio gate):')
for p in sorted(_rejected.tolist()):
    m = p / _fs * 1000.0
    flag = '  <-- INSIDE sine region' if _lo_f <= p < _hi_f else ''
    print(f'  frame {p:4d}  t={m:7.1f} ms  dz={dz_max[p]:7.2f}  '
          f'ratio={_ratio.get(int(p), float("nan")):5.2f}{flag}')
print()

if _kept_arr.size >= 2:
    _gaps_ms = np.diff(_kept_arr) / _fs * 1000.0
    print(f'pulse_train_max_gap_ms = {_cfg.pulse_train_max_gap_ms:.0f} ms  (gaps <= this get bridged)')
    for a, b, g in zip(_kept_arr[:-1], _kept_arr[1:], _gaps_ms):
        bridged = '  BRIDGED' if g <= _cfg.pulse_train_max_gap_ms else ''
        straddle = ('  straddles sine region'
                    if (a < _hi_f and b > _lo_f and not (_lo_f <= a < _hi_f and _lo_f <= b < _hi_f))
                    else '')
        print(f'  {a/_fs*1000:6.1f} -> {b/_fs*1000:6.1f} ms  gap={g:6.1f} ms{bridged}{straddle}')

# ---- plot -----------------------------------------------------------------
fig_d, axd = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True,
                          gridspec_kw={'height_ratios': [1.2, 1.2, 1.0, 0.35],
                                       'hspace': 0.35})
axd[0].plot(t_ms_, z_L, color='#A855F7', lw=0.8, label='WingL_V13 z')
axd[0].plot(t_ms_, z_R, color='#38BDF8', lw=0.8, label='WingR_V13 z')
axd[0].axvspan(_LO_MS, _HI_MS, color='#E76F51', alpha=0.08, zorder=0)
axd[0].set_ylabel('wing z (mm)')
axd[0].set_title(f'Pair {example_idx} — diagnostic for 225-375 ms misclassification')
axd[0].legend(loc='upper right', fontsize=8)

axd[1].plot(t_ms_, dz_max, color='#333', lw=0.7, label='max(|dZ/dt|_L, |dZ/dt|_R)')
axd[1].axhline(_cfg.pulse_detect_height, color='#999', ls='--', lw=0.8,
               label=f'pulse_detect_height={_cfg.pulse_detect_height:g}')
if _kept_arr.size:
    axd[1].scatter(_kept_arr / _fs * 1000.0, dz_max[_kept_arr],
                   s=32, facecolor='#E76F51', edgecolor='black', zorder=5,
                   label=f'kept (n={_kept_arr.size})')
    for p in _kept_arr:
        axd[1].annotate(f'{_ratio.get(int(p), float("nan")):.1f}',
                        (p / _fs * 1000.0, dz_max[p]),
                        textcoords='offset points', xytext=(3, 5),
                        fontsize=7, color='#E76F51')
if _rejected.size:
    axd[1].scatter(_rejected / _fs * 1000.0, dz_max[_rejected],
                   s=28, facecolor='none', edgecolor='#E76F51', zorder=5,
                   label=f'rejected (n={_rejected.size})')
axd[1].axvspan(_LO_MS, _HI_MS, color='#E76F51', alpha=0.08, zorder=0)
axd[1].set_ylabel('|dz/dt| (mm/s)')
axd[1].legend(loc='upper right', fontsize=8)

_wf = _side['window_features']
_wc_ms = np.asarray(_wf['window_centers']) / _fs * 1000.0
ax2 = axd[2]
ax2.plot(_wc_ms, _wf['peak_freq'], color='#1f77b4', lw=0.9, label='peak_freq (Hz)')
ax2.axhline(_cfg.pulse_peak_freq_min, color='#1f77b4', ls='--', lw=0.8,
            label=f'pulse_peak_freq_min={_cfg.pulse_peak_freq_min:g}')
ax2.set_ylabel('freq (Hz)', color='#1f77b4')
ax2b = ax2.twinx()
ax2b.plot(_wc_ms, _wf['pulse_freq_ratio'], color='#d62728', lw=0.9,
          label='pulse_freq_ratio')
ax2b.axhline(_cfg.pulse_freq_ratio_min, color='#d62728', ls='--', lw=0.8,
             label=f'pulse_freq_ratio_min={_cfg.pulse_freq_ratio_min:g}')
ax2b.set_ylabel('ratio', color='#d62728')
ax2.axvspan(_LO_MS, _HI_MS, color='#E76F51', alpha=0.08, zorder=0)
ax2.legend(loc='upper left', fontsize=7)
ax2b.legend(loc='upper right', fontsize=7)

_lbl_colors = {'pulse': '#E76F51', 'sine': '#2A9D8F',
               'waggle': '#E9C46A', 'quiet': '#cccccc'}
_labels = np.asarray(_side['frame_labels'])
for _k, _c in _lbl_colors.items():
    _m = (_labels == _k)
    if not _m.any():
        continue
    _runs = np.diff(np.concatenate(([0], _m.view(np.int8), [0])))
    _starts = np.flatnonzero(_runs == 1)
    _ends   = np.flatnonzero(_runs == -1)
    for s_, e_ in zip(_starts, _ends):
        axd[3].axvspan(s_ / _fs * 1000.0, e_ / _fs * 1000.0,
                       color=_c, alpha=0.9, ymin=0, ymax=1)
axd[3].set_yticks([])
axd[3].set_ylabel('final label', fontsize=8)
axd[3].set_xlabel('time (ms)')

for _ax in axd:
    _ax.set_xlim(150, 500)
fig_d.tight_layout()
plt.show()


In [ ]:
# --- Figure 3: Walking + COM height during song (same example bout) ---
# Inherits START_FRAME / END_FRAME from the Figure 2 cell above.
# Plotted arrays are sliced to the sub-range so y-axes auto-scale only to
# the in-range data.
ex = results[example_idx]
fs = loc_cfg.fs
T = ex['T']
t_full = np.arange(T) / fs
labels_full = np.asarray(ex['male_labels'])

# Resolve optional sub-range (bout-relative frame indices).
_sf = 0 if START_FRAME is None else max(0, int(START_FRAME))
_ef = T if END_FRAME   is None else min(T, int(END_FRAME))
if _ef <= _sf:
    raise ValueError(f'END_FRAME ({END_FRAME}) must be greater than START_FRAME ({START_FRAME})')
_trimmed = not (_sf == 0 and _ef == T)
clip_tag = f'_f{_sf}-{_ef}' if _trimmed else ''

# Slice every plotted quantity to the sub-range.
t      = t_full[_sf:_ef]
labels = labels_full[_sf:_ef]
forward = np.asarray(ex['kin'].get('forward_speed_bl',
                                    ex['kin']['forward_speed']))[_sf:_ef]
turn    = np.asarray(ex['kin']['turn_rate'])[_sf:_ef]
com_z   = np.asarray(ex['com_z'])[_sf:_ef]

def _shade_segments(ax, t, labels):
    """Shade the axes background by per-frame label runs."""
    n = len(labels)
    i = 0
    while i < n:
        j = i
        while j < n and labels[j] == labels[i]:
            j += 1
        ax.axvspan(t[i], t[min(j, n - 1)], color=SONG_COLORS[labels[i]],
                   alpha=0.18, lw=0)
        i = j

fig, axes = plt.subplots(3, 1, figsize=(7.0, 4.5), sharex=True)
_shade_segments(axes[0], t, labels)
axes[0].plot(t, forward, lw=0.6, color='k')
axes[0].set_ylabel('fwd speed\n(bl/s)')
_range_str = f'  [frames {_sf}-{_ef})' if _trimmed else ''
axes[0].set_title(f'A  Forward speed — pair {example_idx}{_range_str}')

_shade_segments(axes[1], t, labels)
axes[1].plot(t, com_z, lw=0.6, color='k')
axes[1].set_ylabel('COM Z\n(above floor)')
axes[1].set_title('B  COM height above floor')

_shade_segments(axes[2], t, labels)
axes[2].plot(t, turn, lw=0.6, color='k')
axes[2].set_ylabel('turn rate\n(deg/s)')
axes[2].set_xlabel('time (s)')
axes[2].set_title('C  Turn rate')

# Explicit xlim to the trimmed window (kept in sync with the slice).
for _ax in axes:
    _ax.set_xlim(t[0], t[-1])

from matplotlib.patches import Patch
handles = [Patch(color=SONG_COLORS[k], label=k, alpha=0.5)
            for k in ('pulse', 'sine', 'waggle', 'quiet')]
fig.legend(handles=handles, loc='upper right', frameon=False,
            bbox_to_anchor=(0.98, 0.99), ncol=4)
plt.tight_layout()

out = FIG_DIR / f'fig3_walking_example{clip_tag}.pdf'
plt.savefig(out)
print(f'saved {out}')
plt.show()


In [ ]:
# --- Figure 2 multi-page PDF: one page per pair ---
# Generates fig2_song_all_pairs.pdf — same layout as the single-pair Figure 2
# above, full bout per page, for easy scanning. Reuses the helpers defined in
# the Figure 2 cell (_shade_song, _add_segment_freq_annotations, etc.), so
# that cell must have been run first.
from matplotlib.backends.backend_pdf import PdfPages

_fs_song = song_cfg.fs


def _build_song_figure_for_pair(result):
    """Build the Figure-2 layout for one pair over the full bout. Returns fig."""
    ms = result['song0'] if result['sex']['male_id'] == 'fly0' else result['song1']
    idx = result['pair_idx']
    fidx = result.get('filtered_idx')

    T_ = ms['summary']['n_frames']
    t_ms_ = np.arange(T_) / _fs_song * 1000.0

    dom_ = ms['dominant_wing']
    dom_side_ = ms['sides'][dom_]
    wf_ = dom_side_['window_features']
    s_ = dom_side_['summary']

    mode_parts_ = []
    for k, lab in (('frac_pulse', 'pulse'), ('frac_sine', 'sine'),
                   ('frac_waggle', 'waggle'), ('frac_quiet', 'quiet')):
        v = s_.get(k, 0.0)
        if v > 0:
            mode_parts_.append(f'{lab}={100*v:.0f}%')
    mode_str_ = '  '.join(mode_parts_)

    joints_ = ms.get('joints')
    nrows_ = 3 if joints_ is not None else 2
    fig_, axes_ = plt.subplots(
        nrows_, 1, figsize=(10.0, 3.0 * nrows_), sharex=True,
        gridspec_kw={'height_ratios': [1.2, 1.0] + ([1.2] if nrows_ == 3 else []),
                     'hspace': 0.28},
    )
    if nrows_ == 1:
        axes_ = [axes_]
    _pair_label = (f'filtered_idx {fidx} (orig pair_idx {idx})'
                   if fidx is not None else f'pair {idx}')
    fig_.suptitle(
        f'{_pair_label}  ({result["key0"]}/{result["key1"]})  |  '
        f'{ms["summary"]["duration_s"]:.3f}s  |  male={result["sex"]["male_id"]}  |  '
        f'dom_wing={dom_}  |  {mode_str_}',
        fontsize=10, fontweight='bold', y=0.995,
    )

    wd_ = ms['wing_data']

    # Panel 1: Wing tip Z
    ax = axes_[0]
    _shade_song(ax, ms, _fs_song, set())
    ax.plot(t_ms_, np.asarray(wd_['WingL_V13']['z']),
            color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
    ax.plot(t_ms_, np.asarray(wd_['WingR_V13']['z']),
            color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
    _add_segment_freq_annotations(ax, dom_side_['segments'], wf_, _fs_song)
    ax.set_ylabel('Z (world)')
    ax.set_title('Wing tip Z (V13)')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
    ax.grid(True, alpha=0.3)

    # Panel 2: Wing tip Y
    ax = axes_[1]
    _shade_song(ax, ms, _fs_song, set())
    ax.plot(t_ms_, np.asarray(wd_['WingL_V13']['y']),
            color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
    ax.plot(t_ms_, np.asarray(wd_['WingR_V13']['y']),
            color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
    ax.set_ylabel('Y (world)')
    ax.set_title('Wing tip Y (V13)')
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
    ax.grid(True, alpha=0.3)

    # Panel 3 (optional): qpos wing joint angles
    if joints_ is not None:
        ax = axes_[2]
        _shade_song(ax, ms, _fs_song, set())
        _joint_styles = [
            ('yaw_L',   '#7B2D8E', '-',  'L yaw'),
            ('roll_L',  '#7B2D8E', '--', 'L roll'),
            ('pitch_L', '#7B2D8E', ':',  'L pitch'),
            ('yaw_R',   '#0369A1', '-',  'R yaw'),
            ('roll_R',  '#0369A1', '--', 'R roll'),
            ('pitch_R', '#0369A1', ':',  'R pitch'),
        ]
        for jkey, jcolor, jls, jlabel in _joint_styles:
            if jkey in joints_:
                ax.plot(t_ms_, np.degrees(np.asarray(joints_[jkey])),
                        color=jcolor, ls=jls, lw=1.0, alpha=0.85, label=jlabel)
        ax.set_ylabel('angle (deg)')
        ax.set_title('Wing joint angles (qpos)')
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=2)
        ax.grid(True, alpha=0.3)

    axes_[-1].set_xlabel('time (ms)')

    # Pulse event overlay
    _pef = np.asarray(wf_.get('pulse_event_frames', np.array([], dtype=int)))
    if _pef.size > 0:
        _pef_ms = _pef / _fs_song * 1000.0
        for _ax in axes_:
            for i_, xm_ in enumerate(_pef_ms):
                _ax.axvline(xm_, color='#E76F51', lw=0.6, alpha=0.55,
                            ls='--', zorder=4,
                            label=(f'pulse events (n={len(_pef_ms)})'
                                   if i_ == 0 and _ax is axes_[0] else None))
        axes_[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7)

    return fig_


_out_pdf = FIG_DIR / 'fig2_song_all_pairs.pdf'
_n = len(results)
print(f'writing {_n} pair figures to {_out_pdf}')
with PdfPages(_out_pdf) as _pdf:
    for _i, _r in enumerate(results):
        try:
            _fig = _build_song_figure_for_pair(_r)
            _pdf.savefig(_fig, bbox_inches='tight')
            plt.close(_fig)
        except Exception as _e:
            print(f'  pair {_r.get("pair_idx", _i)}: {type(_e).__name__}: {_e}')
        if (_i + 1) % 25 == 0 or (_i + 1) == _n:
            print(f'  wrote {_i + 1}/{_n}')
print(f'saved {_out_pdf}')


In [ ]:
# --- Video overlay: project 3D keypoints onto Cam2012630 frames ---
# Standalone cell — loads h5 + calibration directly, no dependence on the
# `data` / `results` variables. Defaults to pair_idx 40/41 of the Session0 h5
# (absolute video frames near 446547 / 446898 — the two wing-tip identity
# flickers seen in the diagnostic cell).
import csv as _csv
import cv2
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SESSION_DIR = Path(
    '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/'
    '2025_10_20_13_20_04'
)
H5_PATH = (SESSION_DIR / 'Predictions_3D_34643097' /
           'Predictions_3D_34653429' / 'v1' /
           'ik_output_combined_v1_courtship_both.h5')
CAM = 'Cam2012630'
BOUT0, BOUT1 = 'bout_040', 'bout_041'      # pair 20 of Session0
LOCAL_FRAMES = [241, 592]                   # ID-flicker candidates
CROP_PAD = 25                               # px padding around projected kp bbox

with h5py.File(H5_PATH, 'r') as _f:
    _kp0 = np.asarray(_f[f'{BOUT0}/kp_data'])
    _kp1 = np.asarray(_f[f'{BOUT1}/kp_data'])
    _sf  = int(_f[f'info/start_frames/{int(BOUT0.split("_")[1])}'][()])
    _kn  = _f['info/kp_names']
    _KP_NAMES = [_kn[k][()].decode() for k in sorted(_kn.keys(), key=int)]
if _kp0.ndim == 2:
    _kp0 = _kp0.reshape(_kp0.shape[0], -1, 3)
    _kp1 = _kp1.reshape(_kp1.shape[0], -1, 3)

with open(SESSION_DIR / 'calibration' / f'{CAM}_dlt.csv') as _f:
    _L = np.array([float(r[0]) for r in _csv.reader(_f)])

def _dlt_project(L, X):
    d = L[8]*X[..., 0] + L[9]*X[..., 1] + L[10]*X[..., 2] + 1.0
    u = (L[0]*X[..., 0] + L[1]*X[..., 1] + L[2]*X[..., 2] + L[3]) / d
    v = (L[4]*X[..., 0] + L[5]*X[..., 1] + L[6]*X[..., 2] + L[7]) / d
    return np.stack([u, v], axis=-1)

def _build_edges(names):
    pairs = [
        ('Antenna_Base', 'Scutellum'),
        ('EyeL', 'Antenna_Base'), ('EyeR', 'Antenna_Base'),
        ('Scutellum', 'Abd_A4'), ('Abd_A4', 'Abd_tip'),
        ('Scutellum', 'WingL_base'), ('WingL_base', 'WingL_V12'),
        ('WingL_V12', 'WingL_V13'),
        ('Scutellum', 'WingR_base'), ('WingR_base', 'WingR_V12'),
        ('WingR_V12', 'WingR_V13'),
    ]
    for side in ('L', 'R'):
        for leg in ('T1', 'T2', 'T3'):
            chain = [f'{leg}{side}_ThxCx', f'{leg}{side}_Tro',
                     f'{leg}{side}_FeTi', f'{leg}{side}_TiTa',
                     f'{leg}{side}_TaT1', f'{leg}{side}_TaT3',
                     f'{leg}{side}_TaTip']
            if chain[0] not in names:
                chain = ['Scutellum'] + chain[1:]
            for a, b in zip(chain[:-1], chain[1:]):
                if a in names and b in names:
                    pairs.append((a, b))
    return [(names.index(a), names.index(b)) for a, b in pairs
            if a in names and b in names]

_EDGES = _build_edges(_KP_NAMES)

def _draw_overlay(ax, img, p0, p1, edges, pad=CROP_PAD):
    ax.imshow(img)
    for pts, color, lbl in (
        (p0, '#E76F51', 'fly0'), (p1, '#2A9D8F', 'fly1')
    ):
        for a, b in edges:
            ax.plot([pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]],
                    color=color, lw=0.9, alpha=0.85)
        ax.scatter(pts[:, 0], pts[:, 1], s=14, c=color,
                   edgecolor='black', lw=0.3, label=lbl, zorder=5)
    allp = np.concatenate([p0, p1], axis=0)
    u0, v0 = np.nanmin(allp[:, 0]), np.nanmin(allp[:, 1])
    u1, v1 = np.nanmax(allp[:, 0]), np.nanmax(allp[:, 1])
    ax.set_xlim(u0 - pad, u1 + pad)
    ax.set_ylim(v1 + pad, v0 - pad)    # matplotlib image coords (y-down)
    ax.legend(loc='upper right', fontsize=8)

_cap = cv2.VideoCapture(str(SESSION_DIR / f'{CAM}.mp4'))
fig_ovr, axes_ovr = plt.subplots(
    1, len(LOCAL_FRAMES),
    figsize=(6.5 * len(LOCAL_FRAMES), 5.0),
    squeeze=False,
)
for _ax, _lf in zip(axes_ovr[0], LOCAL_FRAMES):
    _abs = _sf + _lf
    _cap.set(cv2.CAP_PROP_POS_FRAMES, _abs)
    _ok, _fr = _cap.read()
    if not _ok:
        _ax.text(0.5, 0.5, f'frame {_abs} read failed',
                 transform=_ax.transAxes, ha='center')
        continue
    _img = cv2.cvtColor(_fr, cv2.COLOR_BGR2RGB)
    _p0 = _dlt_project(_L, _kp0[_lf])
    _p1 = _dlt_project(_L, _kp1[_lf])
    _draw_overlay(_ax, _img, _p0, _p1, _EDGES)
    _ax.set_title(f'{CAM}  {BOUT0}/{BOUT1}  local={_lf}  '
                  f't={_lf/800*1000:.0f} ms  abs={_abs}',
                  fontsize=10)
_cap.release()
fig_ovr.tight_layout()
plt.show()


In [ ]:
# --- Figure 4: Song-conditioned locomotion across ALL bouts ---
# Concatenate per-frame male metrics across bouts, keeping only frames
# where the male is valid (colocation-filtered already).
agg = defaultdict(lambda: defaultdict(list))  # metric → song → [values]
for r in results:
    valid = np.asarray(r['male_valid'], dtype=bool)
    if not valid.any():
        continue
    labels = np.asarray(r['male_labels'])[valid]
    metric_vals = {
        'speed_bl':  np.asarray(r['kin'].get('speed_bl', r['kin']['speed']))[valid],
        'turn_rate': np.asarray(r['kin']['turn_rate'])[valid],
        'com_z':     np.asarray(r['com_z'])[valid],
    }
    for m, v in metric_vals.items():
        for song in ('pulse', 'sine', 'waggle', 'quiet'):
            sel = (labels == song)
            if not sel.any():
                continue
            vs = v[sel]
            vs = vs[np.isfinite(vs)]
            if vs.size:
                agg[m][song].extend(vs.tolist())

METRIC_LABELS = {
    'speed_bl':  'speed (bl/s)',
    'turn_rate': 'turn rate (deg/s)',
    'com_z':     'COM Z (above floor)',
}
SONG_ORDER = ['pulse', 'sine', 'waggle', 'quiet']

fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.8))
for ax, m in zip(axes, ['speed_bl', 'turn_rate', 'com_z']):
    datas = [np.asarray(agg[m].get(s, []), dtype=float) for s in SONG_ORDER]
    datas = [d[np.isfinite(d)] for d in datas]
    has_all = all(d.size > 10 for d in datas)
    if has_all:
        parts = ax.violinplot(datas, positions=range(len(SONG_ORDER)),
                              widths=0.8, showmeans=True, showextrema=False)
        for pc, s in zip(parts['bodies'], SONG_ORDER):
            pc.set_facecolor(SONG_COLORS[s]); pc.set_alpha(0.7)
            pc.set_edgecolor('k'); pc.set_linewidth(0.4)
    else:
        means = [float(np.mean(d)) if d.size else float('nan') for d in datas]
        sems  = [float(np.std(d) / np.sqrt(max(1, d.size))) if d.size else 0.0
                 for d in datas]
        ax.bar(range(len(SONG_ORDER)), means, yerr=sems,
               color=[SONG_COLORS[s] for s in SONG_ORDER],
               edgecolor='k', linewidth=0.4)
    ax.set_xticks(range(len(SONG_ORDER)))
    ax.set_xticklabels(SONG_ORDER, rotation=0)
    ax.set_ylabel(METRIC_LABELS[m])

fig.suptitle('Song-conditioned male locomotion (all bouts)', y=1.02)
plt.tight_layout()
out = FIG_DIR / 'fig4_agg_song_locomotion.pdf'
plt.savefig(out)
print(f'saved {out}')
plt.show()


In [ ]:
# --- Figure 5: Per-bout song breakdown (stacked bars) ---
df_sorted = df.sort_values(
    by=['frac_pulse', 'frac_sine', 'frac_waggle'], ascending=False
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.0, 2.8))
x = np.arange(len(df_sorted))
bottom = np.zeros(len(df_sorted))
for song in ('pulse', 'sine', 'waggle', 'quiet'):
    vals = df_sorted[f'frac_{song}'].values
    ax.bar(x, vals, bottom=bottom, color=SONG_COLORS[song],
            width=1.0, edgecolor='none', label=song)
    bottom += vals
ax.set_xlim(-0.5, len(df_sorted) - 0.5)
ax.set_ylim(0, 1.0)
ax.set_xlabel('paired bout (sorted by song activity)')
ax.set_ylabel('fraction of male frames')
ax.set_title('Per-bout song breakdown')
ax.legend(frameon=False, loc='upper right', ncol=4, bbox_to_anchor=(1.0, 1.15))
plt.tight_layout()

out = FIG_DIR / 'fig5_per_bout_stacked.pdf'
plt.savefig(out)
print(f'saved {out}')
plt.show()


In [ ]:
# --- Figure 6: Pulse-mode discovery (Pslow vs Pfast, Clemens 2018 analog) ---
# Pools every per-pulse waveform across pairs (male dominant wing), runs
# PCA + 2-component GMM, assigns Pslow/Pfast by cluster mean symmetry,
# and renders a 4-panel composite:
#   A  PCA scatter colored by carrier frequency (spectral center-of-mass)
#   B  PCA scatter colored by symmetry index
#   C  PCA scatter colored by wing extension angle
#   D  Mean ± std waveform per pulse type
from utils.pulse_types import PulseTypeConfig, fit_pulse_type_model, classify_pulses

def _male_pulse_side(r):
    male_song = r['song0'] if r['sex']['male_id'] == 'fly0' else r['song1']
    dw = str(male_song.get('dominant_wing', 'L')).upper()
    side_key = 'L' if dw.startswith('L') else 'R'
    return male_song, side_key


waves_list, sym_list, hz_list, ang_list = [], [], [], []
pair_idx_list, frame_list = [], []
for r in results:
    male_song, side_key = _male_pulse_side(r)
    pf = male_song['sides'][side_key].get('pulse_features')
    if pf is None or pf['waveforms'].shape[0] == 0:
        continue
    w = pf['waveforms']
    waves_list.append(w)
    sym_list.append(pf['symmetry'])
    hz_list.append(pf['carrier_hz'])
    wa = pf['wing_angle']
    if wa is None:
        wa = np.full(w.shape[0], np.nan)
    ang_list.append(np.asarray(wa))
    pair_idx_list.append(np.full(w.shape[0], r.get('pair_idx', -1), dtype=int))
    frame_list.append(np.asarray(pf['peak_frames'], dtype=int))

if not waves_list:
    print('fig6: no pulses in results; skipping')
else:
    W = np.vstack(waves_list)
    sym = np.concatenate(sym_list)
    hz  = np.concatenate(hz_list)
    ang = np.concatenate(ang_list)
    pair_vec = np.concatenate(pair_idx_list)
    print(f'fig6: pooled {W.shape[0]} pulses from {len(waves_list)} pairs, '
          f'window {W.shape[1]} samples')

    pt_cfg = PulseTypeConfig(n_pca=6)
    model = fit_pulse_type_model(W, sym, pt_cfg)
    labels = classify_pulses(W, model)
    print(f'fig6: component_symmetry={model.component_symmetry}  '
          f'label_map={model.label_map}')
    print(f'fig6: Pslow={int((labels=="Pslow").sum())}, '
          f'Pfast={int((labels=="Pfast").sum())}')

    # 2D PCA for scatter (fit fresh on 2 components so we don't discard
    # the GMM's higher-dim model — this is a display projection only).
    from sklearn.decomposition import PCA as _PCA
    pca2 = _PCA(n_components=2, random_state=0).fit(W)
    X2 = pca2.transform(W)

    fig, axes = plt.subplots(2, 2, figsize=(8.0, 7.0), constrained_layout=True)
    axA, axB, axC, axD = axes.flat

    def _scatter(ax, colors, cmap, label, vmin=None, vmax=None):
        sc = ax.scatter(
            X2[:, 0], X2[:, 1],
            c=colors, s=6, cmap=cmap, alpha=0.7, linewidths=0,
            vmin=vmin, vmax=vmax,
        )
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
        cb.set_label(label)

    _scatter(axA, hz, 'viridis', 'carrier (Hz)',
             vmin=np.nanpercentile(hz, 2),
             vmax=np.nanpercentile(hz, 98))
    axA.set_title('A  carrier frequency')

    _scatter(axB, sym, 'coolwarm', 'symmetry',
             vmin=-1.0, vmax=1.0)
    axB.set_title('B  symmetry index')

    ang_plot = np.where(np.isfinite(ang), ang, np.nan)
    _scatter(axC, ang_plot, 'magma', 'wing angle (deg)',
             vmin=np.nanpercentile(ang_plot, 2)
                  if np.isfinite(ang_plot).any() else 0,
             vmax=np.nanpercentile(ang_plot, 98)
                  if np.isfinite(ang_plot).any() else 90)
    axC.set_title('C  wing extension angle')

    # Panel D: mean ± std waveform per type
    fs_axis = np.arange(W.shape[1]) - W.shape[1] // 2
    fs_ms = fs_axis * 1000.0 / song_cfg.fs
    type_colors = {'Pslow': '#1f77b4', 'Pfast': '#d62728'}
    for t in ('Pslow', 'Pfast'):
        sel = labels == t
        if not sel.any():
            continue
        m = W[sel].mean(axis=0)
        s = W[sel].std(axis=0)
        axD.plot(fs_ms, m, color=type_colors[t], lw=1.5,
                 label=f'{t} (n={int(sel.sum())})')
        axD.fill_between(fs_ms, m - s, m + s,
                          color=type_colors[t], alpha=0.2, linewidth=0)
    axD.axhline(0, color='k', lw=0.5, alpha=0.5)
    axD.set_xlabel('time from peak (ms)')
    axD.set_ylabel('wing-Z (z-scored)')
    axD.set_title('D  mean waveform by type')
    axD.legend(loc='upper right', frameon=False)

    fig.suptitle('Figure 6 — Pulse-mode discovery (Pslow vs Pfast)',
                 fontsize=10, y=1.01)
    out = FIG_DIR / 'fig6_pulse_mode_discovery.png'
    fig.savefig(out, dpi=200)
    plt.show()
    print(f'fig6 → {out}')

    # Stash for Fig 7 / Fig 8 cells.
    FIG6_CTX = {
        'W': W,
        'sym': sym,
        'hz': hz,
        'ang': ang,
        'pair_idx': pair_vec,
        'frames': np.concatenate(frame_list),
        'labels': labels,
        'model': model,
    }


In [ ]:
# --- Figure 7: P(Pslow | male↔female distance), Clemens Fig 3D analog ---
# Per-pulse distance between male and female scutellum at the pulse
# frame, binned by body-length-normalised distance.
try:
    ctx = FIG6_CTX
except NameError:
    print('fig7: FIG6_CTX missing (run Fig 6 first); skipping')
    ctx = None

if ctx is not None:
    scut_i = kp_names.index('Scutellum')

    # Per-pulse distance lookup using results list + in-memory data.
    # pair_idx → (male_key, female_key, body_length)
    pair_info = {}
    for r in results:
        mid = r['sex']['male_id']
        male_key = r['key0'] if mid == 'fly0' else r['key1']
        fem_key  = r['key1'] if mid == 'fly0' else r['key0']
        bl = r['sex'].get('body_length_male', np.nan)
        pair_info[r.get('pair_idx', -1)] = (male_key, fem_key, float(bl))

    # Cache per-pair per-frame distance traces (avoid re-reading h5
    # for every pulse).
    dist_cache = {}
    def _get_dist(pi):
        if pi in dist_cache:
            return dist_cache[pi]
        male_key, fem_key, bl = pair_info[pi]
        mk = np.asarray(data[male_key]['kp_data'])
        fk = np.asarray(data[fem_key]['kp_data'])
        if mk.ndim == 2:
            mk = mk.reshape(mk.shape[0], -1, 3)
        if fk.ndim == 2:
            fk = fk.reshape(fk.shape[0], -1, 3)
        T_ = min(len(mk), len(fk))
        d = np.linalg.norm(mk[:T_, scut_i] - fk[:T_, scut_i], axis=1)
        if np.isfinite(bl) and bl > 0:
            d = d / bl
            unit = 'body lengths'
        else:
            unit = 'raw units'
        dist_cache[pi] = (d, unit)
        return dist_cache[pi]

    per_pulse_d = np.full(ctx['frames'].shape[0], np.nan)
    unit = 'raw units'
    for i, (pi, f) in enumerate(zip(ctx['pair_idx'], ctx['frames'])):
        d, unit = _get_dist(int(pi))
        if 0 <= f < len(d):
            per_pulse_d[i] = d[f]
    finite = np.isfinite(per_pulse_d)
    print(f'fig7: {finite.sum()}/{len(per_pulse_d)} pulses with distance '
          f'(unit: {unit})')

    # Bin by distance: choose bin edges adaptively from the data range
    if unit == 'body lengths':
        edges = np.arange(0.0, 10.01, 0.5)
    else:
        # raw units — use per-data percentiles
        lo = float(np.nanpercentile(per_pulse_d, 1))
        hi = float(np.nanpercentile(per_pulse_d, 99))
        edges = np.linspace(lo, hi, 16)
    centers = 0.5 * (edges[:-1] + edges[1:])

    labels = ctx['labels']
    rng = np.random.default_rng(0)
    n_boot = 1000

    frac_mean = np.full(len(centers), np.nan)
    frac_lo   = np.full(len(centers), np.nan)
    frac_hi   = np.full(len(centers), np.nan)
    n_in_bin  = np.zeros(len(centers), dtype=int)
    for bi in range(len(centers)):
        sel = finite & (per_pulse_d >= edges[bi]) & (per_pulse_d < edges[bi + 1])
        n_in_bin[bi] = int(sel.sum())
        if n_in_bin[bi] < 5:
            continue
        y = (labels[sel] == 'Pslow').astype(float)
        frac_mean[bi] = float(y.mean())
        idx = rng.integers(0, y.size, size=(n_boot, y.size))
        boots = y[idx].mean(axis=1)
        frac_lo[bi] = float(np.percentile(boots, 2.5))
        frac_hi[bi] = float(np.percentile(boots, 97.5))

    fig, ax = plt.subplots(figsize=(5.0, 3.3), constrained_layout=True)
    good = np.isfinite(frac_mean)
    weak = good & (n_in_bin < 20)
    strong = good & (n_in_bin >= 20)
    ax.plot(centers[strong], frac_mean[strong], '-o', color='#1f77b4',
            lw=1.5, ms=4, label='P(Pslow | d)  (n ≥ 20)')
    ax.fill_between(centers[strong], frac_lo[strong], frac_hi[strong],
                    color='#1f77b4', alpha=0.25, linewidth=0,
                    label='95 % bootstrap CI')
    if weak.any():
        ax.plot(centers[weak], frac_mean[weak], 'o', mfc='none',
                mec='#999999', mew=1.0, ms=4, label='n < 20 (weak)')

    ax.axhline(0.5, color='k', lw=0.5, alpha=0.4, linestyle=':')
    ax.set_xlabel(f'male↔female scutellum distance ({unit})')
    ax.set_ylabel('P(Pslow)')
    ax.set_ylim(-0.02, 1.02)
    ax.set_title('Figure 7 — Pulse-type probability vs distance')
    ax.legend(loc='best', frameon=False)

    ax2 = ax.twinx()
    ax2.bar(centers, n_in_bin, width=np.diff(edges),
            color='#bdbdbd', alpha=0.35, align='center',
            edgecolor='none', zorder=0)
    ax2.set_ylabel('pulses per bin', color='#777777')
    ax2.spines['top'].set_visible(False)
    ax2.tick_params(colors='#777777')

    out = FIG_DIR / 'fig7_pslow_fraction_vs_distance.png'
    fig.savefig(out, dpi=200)
    plt.show()
    print(f'fig7 → {out}')


In [ ]:
# --- Figure 8: IPI (inter-pulse interval) histogram ---
# Pools inter-pulse intervals across all pair males' dominant wing, and
# overlays per-type IPIs using the Fig-6 labels. The canonical
# D. melanogaster IPI is ~35 ms.
try:
    ctx = FIG6_CTX
except NameError:
    print('fig8: FIG6_CTX missing (run Fig 6 first); skipping')
    ctx = None

if ctx is not None:
    # Collect per-pair male IPIs (all pulses, not split yet).
    ipi_all = []
    ipi_slow = []
    ipi_fast = []

    # We have labels in ctx indexed by (pair, frame). For per-train IPIs we
    # need consecutive pulses from the same pair — ctx['frames'] is the
    # concatenation of per-pair peak_frames in pair order, so we can split
    # by pair_idx.
    pair_vec = ctx['pair_idx']
    frames   = ctx['frames']
    labels   = ctx['labels']
    fs_ = song_cfg.fs
    unique_pairs = np.unique(pair_vec)
    for pi in unique_pairs:
        sel = pair_vec == pi
        fs_this = frames[sel]
        lbl_this = labels[sel]
        order = np.argsort(fs_this)
        fs_this = fs_this[order]
        lbl_this = lbl_this[order]
        if fs_this.size < 2:
            continue
        ipi = np.diff(fs_this) * 1000.0 / fs_
        ipi_all.append(ipi)
        # Label an IPI by the label of its *leading* pulse.
        for j, iv in enumerate(ipi):
            if lbl_this[j] == 'Pslow':
                ipi_slow.append(iv)
            else:
                ipi_fast.append(iv)

    ipi_all  = np.concatenate(ipi_all)  if ipi_all  else np.array([])
    ipi_slow = np.asarray(ipi_slow, dtype=float)
    ipi_fast = np.asarray(ipi_fast, dtype=float)
    print(f'fig8: {ipi_all.size} IPIs total  '
          f'(Pslow-led {ipi_slow.size}, Pfast-led {ipi_fast.size})')

    fig, ax = plt.subplots(figsize=(5.2, 3.3), constrained_layout=True)
    bins = np.logspace(np.log10(1), np.log10(500), 50)
    if ipi_all.size:
        ax.hist(ipi_all, bins=bins, color='#666666', alpha=0.35,
                label=f'all (n={ipi_all.size})', edgecolor='none')
    if ipi_slow.size:
        ax.hist(ipi_slow, bins=bins, histtype='step',
                color='#1f77b4', lw=1.5,
                label=f'Pslow-led (n={ipi_slow.size})')
    if ipi_fast.size:
        ax.hist(ipi_fast, bins=bins, histtype='step',
                color='#d62728', lw=1.5,
                label=f'Pfast-led (n={ipi_fast.size})')
    ax.axvline(35, color='k', lw=1.0, linestyle='--', alpha=0.6,
               label='canonical 35 ms')
    ax.set_xscale('log')
    ax.set_xlabel('inter-pulse interval (ms)')
    ax.set_ylabel('count')
    ax.set_title('Figure 8 — IPI distribution')
    ax.legend(loc='best', frameon=False)
    out = FIG_DIR / 'fig8_ipi_histogram.png'
    fig.savefig(out, dpi=200)
    plt.show()
    print(f'fig8 → {out}')


In [ ]:
# --- Export per-bout results table ---
csv_path = FIG_DIR / 'per_bout_results.csv'
df.to_csv(csv_path, index=False)
print(f'wrote {csv_path}  ({len(df)} rows, {len(df.columns)} cols)')


In [ ]:
# --- Text summary ---
n_pairs = len(df)
n_conf  = int((df['confidence'] >= 0.5).sum())
n_song  = int((df['criterion'] == 'song_fraction').sum())
n_disag = int(df['disagree_bodylen'].sum())

print(f'pairs processed        : {n_pairs}')
print(f'  male-ID confidence ≥0.5 : {n_conf}  ({100*n_conf/n_pairs:.1f}%)')
print(f'  song-based male ID      : {n_song}  ({100*n_song/n_pairs:.1f}%)')
print(f'  body-length disagreement: {n_disag}  ({100*n_disag/max(1,n_song):.1f}% of song-based calls)')
print()
print('median song fractions (male):')
for s in ('pulse', 'sine', 'waggle', 'quiet'):
    print(f'  {s:7s}: {df[f"frac_{s}"].median():.3f}')
print()
print('median locomotion (male):')
print(f'  mean speed      : {df["mean_speed_bl"].median():.3f} bl/s')
print(f'  mean COM Z      : {df["mean_com_z"].median():.3f}')
print(f'  walking fraction: {df["walking_fraction"].median():.3f}')


# Figure 6 — Consolidated courtship figure

Combines the EXAMPLE_IDX=65 exemplar bout (rows 1–3) with population summaries
(row 4). Layout is built by `utils.courtship_figure_panels.assemble_figure` and
each panel is filled by an independent `panel_*` function so any panel can be
swapped out without touching the others.

Fill in the three placeholders below before running:
- `CAM_MP4_PATH` — path to the per-camera mp4 covering this bout
- `ROI`         — (x, y, w, h) crop in pixels
- `FREE_WALK_H5_PATH` — path to a non-courtship free-walking combined h5


In [ ]:
by_orig = {r['pair_idx']: r for r in results}


In [ ]:
# --- Figure 6: Reorganized consolidated courtship figure (183 mm × 130 mm) ---
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt

from utils import courtship_figure_panels as cfp
from utils import courtship_loader as _cl
from utils import pulse_type_cache as _ptc
from utils import free_walking_loader as _fwl

importlib.reload(cfp)
importlib.reload(_cl)
importlib.reload(_ptc)
importlib.reload(_fwl)
get_pulse_type_labels = _ptc.get_pulse_type_labels
load_free_walking_scutellum_z = _fwl.load_free_walking_scutellum_z
load_orig_keypoints_index = _cl.load_orig_keypoints_index
get_orig_keypoints_for_combined_bout = _cl.get_orig_keypoints_for_combined_bout

# ------------------------------------------------------------------ inputs
# --- Video (rows 1+2) -------------------------------------------------
CAM_MP4_PATH  = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Cam2012630.mp4"
DLT_CSV_PATH  = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/calibration/Cam2012630_dlt.csv"
# CAM_MP4_PATH      = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Cam2012630.mp4"
# DLT_CSV_PATH      = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/calibration/Cam2012630_dlt.csv"
CROP_WH           = (192, 192)                      # (w, h) per-frame crop centered on the flies
KP_LABEL_FRAME    = None                            # None = middle of strip
KP_LABEL_OFFSETS  = {                               # text nudge in cropped px
    "Scutellum":  (0,0),#( 8, -4),
    "WingL_V13":  (0,0),#(-12, -10),
    "WingR_V13":  (0,0),#( 12, -10),
}
KP_SCALE         = 0.1                              # 3D unit -> DLT unit (mm -> cm here)

# Original (pre-rescale) keypoints — required for correct DLT projection.
# Combined h5 stores the body-model-rescaled kp_data; orig_keypoints live
# in each per-prediction `preprocessing/preprocessed_bout_*_both.h5`.
ORIG_KP_SEARCH_DIR = "/data2/users/eabe/datasets/Johnson_lab/courtship/04092026_bouts_04112026"

# --- Free-walking dataset (row 3 right) --------------------------------
FREE_WALK_H5_PATH = "/data2/users/eabe/datasets/Johnson_lab/free_walking/Data_analysis/analysis/v1/ik_output_combined_v1_free_walking.h5"

# --- Render (row 4): male + female + floor -----------------------------
VIZ_MODEL_XML    = "/home/eabe/Research/MyRepos/fruitfly_body_models/fruitfly_v1/fruitfly_v1_free.xml"
VIZ_FLOOR_XML    = "/home/eabe/Research/MyRepos/fruitfly_body_models/fruitfly_v1/floor.xml"
VIZ_SETTINGS_FLY0 = "Earthy_V1_courtship_fly0"
VIZ_SETTINGS_FLY1 = "Earthy_V1_courtship_fly1"
VIZ_CAMERA       = "track1_fly0"
VIZ_HEIGHT       = 512
VIZ_WIDTH        = 512
RENDER_CROP_WH   = (192, 192)                      # (w, h) centered post-render crop
RENDER_CROP      = (VIZ_WIDTH // 2 - RENDER_CROP_WH[0] // 2,
                    VIZ_HEIGHT // 2 - RENDER_CROP_WH[1] // 2,
                    RENDER_CROP_WH[0], RENDER_CROP_WH[1])

# --- Exemplar bout + frame selection -----------------------------------
EXAMPLE_IDX     = 12  # bout from 2026_04_02_12_11_50 (matches CAM_MP4_PATH); previously 132
START_FRAME     = int((0   / 1000) * 800)
END_FRAME       = None # int((800 / 1000) * 800)           # None => full clip
N_FRAMES_STRIP  = 6
FRAME_INDICES   = None                              # None = evenly spaced

# ------------------------------------------------------------------ slice
fs = song_cfg.fs
ex = results[EXAMPLE_IDX]
clip_T = int(ex["T"])
if END_FRAME is None:
    END_FRAME = clip_T
END_FRAME = min(int(END_FRAME), clip_T)
START_FRAME = max(0, int(START_FRAME))
sel = slice(START_FRAME, END_FRAME)

wing_data   = ex["song0"]["wing_data"]
wingL_z     = wing_data["WingL_V13"]["z"][sel]
wingR_z     = wing_data["WingR_V13"]["z"][sel]
scutellum_z = ex["com_z"][sel]
seg_L       = ex["song0"]["sides"]["L"]["segments"]
seg_R       = ex["song0"]["sides"]["R"]["segments"]
t_ms        = (np.arange(START_FRAME, END_FRAME) / fs) * 1000.0
if FRAME_INDICES is not None:
    frame_idx_strip = np.asarray(FRAME_INDICES, dtype=int)
else:
    frame_idx_strip = np.linspace(START_FRAME, END_FRAME - 1,
                                  N_FRAMES_STRIP, dtype=int)
N_FRAMES_STRIP = len(frame_idx_strip)

# Per-frame extended vs folded wing for the exemplar bout (Row 3 left).
angle_L = np.asarray(ex["song0"]["angle_L"])
angle_R = np.asarray(ex["song0"]["angle_R"])
zL_full = np.asarray(wing_data["WingL_V13"]["z"])
zR_full = np.asarray(wing_data["WingR_V13"]["z"])
ext_is_L = angle_L > angle_R
ext_z_w  = np.where(ext_is_L, zL_full, zR_full)[sel]
fold_z_w = np.where(ext_is_L, zR_full, zL_full)[sel]

# ------------------------------------------------------------------ aggregates
# Per-bout MEAN scutellum z by song state (Row 3 right; n = bouts, not frames).
pulse_z_list, sine_z_list = [], []
for r in results:
    v = np.asarray(r["male_valid"], dtype=bool)
    lab = np.asarray(r["male_labels"])
    z = np.asarray(r["com_z"], dtype=float)
    base = v & np.isfinite(z)
    mp = base & (lab == "pulse")
    ms = base & (lab == "sine")
    if mp.any():
        pulse_z_list.append(float(np.mean(z[mp])))
    if ms.any():
        sine_z_list.append(float(np.mean(z[ms])))
pulse_z = np.asarray(pulse_z_list, dtype=float)
sine_z  = np.asarray(sine_z_list,  dtype=float)

try:
    walking_z = load_free_walking_scutellum_z(FREE_WALK_H5_PATH, per_bout=True)
except (FileNotFoundError, OSError) as e:
    print(f"(free walking h5 not loaded — {e}); using empty array")
    walking_z = np.zeros(0)

# Pooled extended-wing IK yaw angles by song state (Row 3 col 2).
# Yaw is the wing swing DOF (rotation around vertical body axis); rest at
# springref=1.5 rad. Reported in degrees for readability.
WING_IK_DOF = "yaw"   # one of {"yaw", "roll", "pitch"}
ext_pulse_a, ext_sine_a = [], []
for r in results:
    joints = r["song0"].get("joints")
    if joints is None:
        continue
    yL = np.degrees(np.asarray(joints[f"{WING_IK_DOF}_L"], dtype=float))
    yR = np.degrees(np.asarray(joints[f"{WING_IK_DOF}_R"], dtype=float))
    lab = np.asarray(r["male_labels"])
    v   = np.asarray(r["male_valid"], dtype=bool)
    ext_is_L_r = np.abs(yL) > np.abs(yR)
    ext_a = np.where(ext_is_L_r, yL, yR)
    base = v & np.isfinite(ext_a)
    mp = base & (lab == "pulse")
    ms = base & (lab == "sine")
    if mp.any(): ext_pulse_a.append(ext_a[mp])
    if ms.any(): ext_sine_a.append(ext_a[ms])
ext_pulse_a = np.concatenate(ext_pulse_a) if ext_pulse_a else np.zeros(0)
ext_sine_a  = np.concatenate(ext_sine_a)  if ext_sine_a  else np.zeros(0)

# Pooled L vs R wing-z phase difference during sine song (Row 3 col 0 polar).
from scipy.signal import hilbert as _hilbert
phase_diffs = []
for r in results:
    wd = r["song0"]["wing_data"]
    zL_full_r = np.asarray(wd["WingL_V13"]["z"], dtype=float)
    zR_full_r = np.asarray(wd["WingR_V13"]["z"], dtype=float)
    for s in r["song0"]["sides"]["L"]["segments"]:
        if s.get("type") != "sine":
            continue
        i0, i1 = int(s["start"]), int(s["end"]) + 1
        if i1 - i0 < 16:
            continue
        a = zL_full_r[i0:i1]; b = zR_full_r[i0:i1]
        if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):
            continue
        a = a - np.mean(a); b = b - np.mean(b)
        pa = np.angle(_hilbert(a))
        pb = np.angle(_hilbert(b))
        R_seg = np.mean(np.exp(1j * (pa - pb)))
        if not np.isfinite(R_seg):
            continue
        phase_diffs.append(np.angle(R_seg))
phase_diffs = (np.asarray(phase_diffs, dtype=float) if phase_diffs
               else np.zeros(0, dtype=float))

# Pulse-type labels (cached) for wing/scut shading.
pulse_type_results = get_pulse_type_labels(
    results, cache_path=FIG_DIR / "pulse_type_cache.pkl", fs=fs
)
pulse_type_labels = pulse_type_results["labels"]

_ex_pair = int(ex.get("pair_idx", EXAMPLE_IDX))
pulse_type_side = {}
for _side in ("L", "R"):
    _pf = ex["song0"]["sides"].get(_side, {}).get("pulse_features", {}) or {}
    _peaks = np.asarray(_pf.get("peak_frames", np.zeros(0, dtype=int)))
    _labs  = np.asarray(pulse_type_labels.get(_ex_pair, {}).get(_side, np.array([])))
    if _peaks.size and _labs.size == _peaks.size:
        pulse_type_side[_side] = {"peak_frames": _peaks, "labels": _labs}

_dw = str(ex["song0"].get("dominant_wing", "L")).upper()
_dom_side = "L" if _dw.startswith("L") else "R"
_dom_track = pulse_type_side.get(_dom_side, {})

# DLT calibration + ORIGINAL 3D keypoints (un-rescaled) for projection.
dlt_coeffs = cfp._dlt_load(DLT_CSV_PATH)
VIDEO_FRAME_OFFSET = int(info["start_frames"][bout_keys.index(ex["key0"])])
try:
    orig_kp_index = load_orig_keypoints_index(ORIG_KP_SEARCH_DIR)
except (FileNotFoundError, OSError) as e:
    print(f"(orig keypoints not loaded — {e}); falling back to body-model kp_data")
    orig_kp_index = {}
kp_xyz = get_orig_keypoints_for_combined_bout(
    orig_kp_index, info, bout_keys, ex["key0"]
)
_USE_KP_OVERLAY = kp_xyz is not None
if kp_xyz is None:
    print("(no orig_keypoints match — keypoint overlays disabled for this bout)")
    kp_xyz = data[ex["key0"]]["kp_data"]
else:
    print(f"orig_keypoints found: shape {kp_xyz.shape}")

# Per-frame midpoint of the two flies' Scutellum in the world frame used by
# DLT. Body-model-rescaled kp_data is used here (stable across bouts and
# present for every pair); mm-scale matches KP_SCALE.
_scut_idx = kp_names.index("Scutellum")
_kp0 = np.asarray(data[ex["key0"]]["kp_data"])
_kp1 = np.asarray(data[ex["key1"]]["kp_data"])
if _kp0.ndim == 2 and _kp0.shape[-1] != 3:
    _kp0 = _kp0.reshape(_kp0.shape[0], -1, 3)
    _kp1 = _kp1.reshape(_kp1.shape[0], -1, 3)
_T_c = min(_kp0.shape[0], _kp1.shape[0], clip_T)
center_xyz = 0.5 * (_kp0[:_T_c, _scut_idx, :] + _kp1[:_T_c, _scut_idx, :])

# ------------------------------------------------------------------ assemble
fig, axd = cfp.assemble_figure(fig_width_mm=183.0, fig_height_mm=130.0,
                               n_frames_strip=N_FRAMES_STRIP)

# Row 1 left — labeled keypoint frame
label_fidx = (KP_LABEL_FRAME if KP_LABEL_FRAME is not None
              else int(frame_idx_strip[len(frame_idx_strip) // 2]))
try:
    if _USE_KP_OVERLAY:
            cfp.panel_kp_label_frame(
                axd["kp_label"], CAM_MP4_PATH, label_fidx,
                kp_xyz=kp_xyz, kp_names=kp_names, dlt_coeffs=dlt_coeffs,
                center_xyz=center_xyz, crop_wh=CROP_WH,
                label_offsets=KP_LABEL_OFFSETS,
                kp_scale=KP_SCALE, video_frame_offset=VIDEO_FRAME_OFFSET,
            )
    else:
        cfp.panel_kp_label_frame(
            axd["kp_label"], CAM_MP4_PATH, label_fidx,
            kp_xyz=kp_xyz, kp_names=kp_names, dlt_coeffs=dlt_coeffs,
            center_xyz=center_xyz, crop_wh=CROP_WH,
            label_offsets=KP_LABEL_OFFSETS,
            kp_scale=KP_SCALE, video_frame_offset=VIDEO_FRAME_OFFSET,
        )
except (FileNotFoundError, OSError) as e:
    print(f"(label frame not loaded — {e}); leaving Row 1 left blank")

# Row 1 right — wing V13 z + scutellum z time series
cfp.panel_wing_z_traces(axd["wing"], t_ms, wingL_z, wingR_z, seg_L, seg_R,
                        fs=fs, frame_range=(START_FRAME, END_FRAME),
                        pulse_type_side=pulse_type_side)
cfp.panel_scutellum_z_trace(axd["scut"], t_ms, scutellum_z,
                            segments=seg_L, fs=fs,
                            frame_range=(START_FRAME, END_FRAME),
                            pulse_peak_frames=_dom_track.get("peak_frames"),
                            pulse_subtype_labels=_dom_track.get("labels"))

# Row 2 — video frames with all keypoints overlayed
try:
    if _USE_KP_OVERLAY:
            cfp.panel_video_strip_with_kp(
                axd["video"], CAM_MP4_PATH, frame_idx_strip[: len(axd["video"])],
                kp_xyz_per_frame=kp_xyz, kp_names=kp_names, dlt_coeffs=dlt_coeffs,
                overlay_kps=None, fs=fs,
                center_xyz=center_xyz, crop_wh=CROP_WH,
                kp_scale=KP_SCALE, video_frame_offset=VIDEO_FRAME_OFFSET,
            )
    else:
        cfp.panel_video_strip(
            axd["video"], CAM_MP4_PATH, frame_idx_strip[: len(axd["video"])],
            fs=fs,
            center_xyz=center_xyz, crop_wh=CROP_WH,
            dlt_coeffs=dlt_coeffs, kp_scale=KP_SCALE,
            video_frame_offset=VIDEO_FRAME_OFFSET,
            )
except (FileNotFoundError, OSError) as e:
    print(f"(video strip not loaded — {e}); leaving Row 2 blank")

# Row 3 — sine in-phase | joint angle density | Pslow/Pfast | scutellum z
sine_segs_window = [s for s in seg_L if s.get("type") == "sine"]

# Zoom window for the sine in-phase panel (ms, relative to bout start).
SINE_PHASE_T_MS = (800.0, 1100.0)
_zf0 = max(START_FRAME, int(round(SINE_PHASE_T_MS[0] * fs / 1000.0)))
_zf1 = min(END_FRAME,   int(round(SINE_PHASE_T_MS[1] * fs / 1000.0)))
_zsel = slice(_zf0 - START_FRAME, _zf1 - START_FRAME)
cfp.panel_sine_wing_inphase(
    axd["sine_phase"], t_ms[_zsel], ext_z_w[_zsel], fold_z_w[_zsel], fs=fs,
    frame_range=(_zf0, _zf1),
    sine_segments=sine_segs_window, title=''
)
cfp.panel_wing_phase_polar(
    axd["wing_phase_polar"], phase_diffs,
)
cfp.panel_joint_angle_density(
    axd["angle_2d"], ext_pulse_a, ext_sine_a,
    range_deg=(-70.0, 70.0),
    title="",
)
cfp.panel_pulse_classification(axd["pulse_class"], pulse_type_results,
                               show_std=True, title='')
cfp.panel_z_height_singing_vs_walking(axd["zheight"], pulse_z, sine_z, walking_z,
                                      kind="box", title='')

# Row 4 — MuJoCo render strip (same frame indices as Row 2)
qpos0 = data[ex["key0"]].get("qpos")
qpos1 = data[ex["key1"]].get("qpos")
if qpos0 is None or qpos1 is None or VIZ_MODEL_XML is None:
    print("(no qpos for one of the flies or no model — leaving Row 4 blank)")
else:
    T_min = min(len(qpos0), len(qpos1), clip_T)
    qpos_pair = np.concatenate(
        [np.asarray(qpos0)[:T_min], np.asarray(qpos1)[:T_min]], axis=-1,
    )
    viz = cfp.build_courtship_pair_visualizer(
        flybody_xml=VIZ_MODEL_XML,
        floor_xml=VIZ_FLOOR_XML,
        settings_fly0=VIZ_SETTINGS_FLY0,
        settings_fly1=VIZ_SETTINGS_FLY1,
    )
    cfp.panel_render_strip(
        axd["render"], qpos_pair, frame_idx_strip,
        viz=viz, camera=VIZ_CAMERA,
        height_px=VIZ_HEIGHT, width_px=VIZ_WIDTH,
        crop=RENDER_CROP,
    )

out_pdf = FIG_DIR / "fig6_consolidated_courtship.pdf"
out_svg = FIG_DIR / "fig6_consolidated_courtship.svg"
fig.savefig(out_pdf)
fig.savefig(out_svg, dpi=300,transparent=True)
print(f"saved {out_pdf}")
print(f"saved {out_svg}")
plt.show()


In [ ]:
info['fly_ids'][EXAMPLE_IDX]

## Figure captions

**Figure 1 — Male/female identification.** (A) Scatter of song fractions for fly0 vs fly1 in every paired bout, colored by which fly was assigned male. Points far from the diagonal reflect confident, song-driven assignments. (B) Body length distribution for the assigned male and female across all pairs (sanity check: males should be the smaller fly). (C) Histogram of male-ID confidence; the vertical dashed line marks the default threshold used for downstream filtering.

**Figure 2 — Song classification (example bout).** (A) Dominant-wing tip Z trace over time for the assigned male. (B) Spectrogram of the same trace with horizontal guides at the waggle / sine / pulse frequency band edges used by the classifier. (C) Per-frame label strip (pulse / sine / waggle / quiet).

**Figure 3 — Walking and COM height during song (same bout).** (A) Male forward speed in body-lengths per second, shaded by the per-frame song label. (B) Male COM Z above the per-bout estimated floor plane. (C) Male turn rate in deg/s.

**Figure 4 — Song-conditioned locomotion (aggregate).** Violin plots (or bars + SEM when sample sizes are small) of male walking speed, turn rate, and COM height, split by song label, concatenated across all bouts with a confident male assignment.

**Figure 5 — Per-bout song breakdown.** Stacked bars showing the fraction of male frames labeled pulse / sine / waggle / quiet for every paired bout, sorted by pulse+sine content. Quiet bouts sit on the right.
